# Used Car Prediction
This notebook develops machine learning models to predict used-car prices based on vehicle characteristics.

In [1]:
import pandas as pd

df = pd.read_csv("../data/cleaned/used_cars_cleaned.csv")

df.head()

,brand,model,model_year,mileage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price,transmission_group
0,Ford,Utility Police Interceptor Base,2013,51000,E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,10300,Automatic
1,Hyundai,Palisade SEL,2021,34742,Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,38005,Automatic
2,Lexus,RX 350 RX 350,2022,22372,Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,Unknown,54598,Automatic
3,INFINITI,Q50 Hybrid Sport,2015,88900,Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,15500,Automatic
4,Audi,Q3 45 S line Premium Plus,2021,9835,Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,Unknown,34999,Automatic


In [2]:
df.columns

Index(['brand', 'model', 'model_year', 'mileage', 'fuel_type', 'engine',
       'transmission', 'ext_col', 'int_col', 'accident', 'clean_title',
       'price', 'transmission_group'],
      dtype='object')

In [3]:
df.shape

(4009, 13)

In [4]:
features = [
    "model_year",
    "mileage",
    "brand",
    "fuel_type",
    "accident",
    "clean_title",
    "transmission_group"
]

X = df[features]
y = df["price"]

X.shape


(4009, 7)

In [5]:
y.shape

(4009,)

In [6]:
X.dtypes

model_year             int64
mileage                int64
brand                 object
fuel_type             object
accident              object
clean_title           object
transmission_group    object
dtype: object

In [7]:
numeric_f = [
    "model_year",
    "mileage"
]

categorical_f = [
    "brand",
    "fuel_type",
    "accident",
    "clean_title",
    "transmission_group"
]


In [8]:
## This step is to change the categorical data into numeric data by using OneHotEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers = [
        (
        "cat_step_name",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_f
        ),
        (
            "num_step_name",
            "passthrough",
            numeric_f
        )
    ]
)

In [9]:
## split data for training and testing
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)


In [10]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (3207, 7)
X_test: (802, 7)
y_train: (3207,)
y_test: (802,)


In [11]:
# Machine Learning Model
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor",LinearRegression())
    ]
)

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_step_name', ...), ('num_step_name', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the d

In [12]:
y_pred = model.predict(X_test)
y_pred[:5]

array([45240.02642979, 20855.38617364, 49106.35374206, 73189.90155856,
       39884.99797303])

In [13]:
# compare y_pred vs y_test
comparison = pd.DataFrame({
    "actual_price": y_test.iloc[:5].values,
    "predicted_price": y_pred[:5]
})

comparison

,actual_price,predicted_price
0,28000,45240.026430
1,5900,20855.386174
2,41000,49106.353742
3,40250,73189.901559
4,77999,39884.997973


In [14]:
## Evaluate the errors of the model
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test,y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R^2:", r2)

MAE: 30131.727518374835
RMSE: 139501.20955271245
R^2: 0.0478964836513438


### Baseline Linear Regression Results

The baseline Linear Regression model produced an MAE of approximately $30,132, and RMSE of $139,501, and R^2 of 0.048

The high prediction errors and low R squared indicate that the model does not capture the pricing patterns well.

In [15]:
results = pd.DataFrame({
    "actual_price": y_test.values,
    "predicted_price": y_pred
})

results["error"] = (
    results["actual_price"] - results["predicted_price"]
)

results["absolute_error"] = results["error"].abs()


In [16]:
results.sort_values(
    "absolute_error",
    ascending=False
).head(10)

,actual_price,predicted_price,error,absolute_error
70,2954083,45348.891725,2.908734e+06,2.908734e+06
165,1950995,53014.464642,1.897981e+06,1.897981e+06
493,1599000,105004.717629,1.493995e+06,1.493995e+06
95,489995,64760.181394,4.252348e+05,4.252348e+05
548,429998,31454.526361,3.985435e+05,3.985435e+05
517,399950,105545.639823,2.944044e+05,2.944044e+05
648,399900,125377.280298,2.745227e+05,2.745227e+05
760,359991,137343.249764,2.226478e+05,2.226478e+05
522,359991,137349.167569,2.226418e+05,2.226418e+05
579,314900,104606.853589,2.102931e+05,2.102931e+05


### Error Analysis

The largest prediction errors occur among extremely high-priced vehicles. For example, several vehicles priced above $1 million were predicted at values closer to typical used car prices. 

In [17]:
# We use log to reduce the gap between luxury cars and typical cars.
import numpy as np

y_log = np.log1p(df["price"])

#split the data again
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X,
    y_log,
    test_size = 0.2,
    random_state = 42
)

model_log = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

model_log.fit(X_train, y_train_log)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_step_name', ...), ('num_step_name', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the d

In [18]:
y_pred_log = model_log.predict(X_test)
## convert the log number into dollars value
y_pred_price = np.expm1(y_pred_log)

y_pred_price[:5]

array([34245.67828605, 17307.56458545, 45487.19064503, 51101.24150249,
       42867.66960008])

In [19]:
## Evaluate again the new prediction after we log the price

y_test_price = np.expm1(y_test_log)

mae_log = mean_absolute_error(y_test_price, y_pred_price)
rmse_log = root_mean_squared_error(y_test_price, y_pred_price)
r2_log = r2_score(y_test_price, y_pred_price)

print("Log Model MAE:", mae_log)
print("Log Model RMSE:", rmse_log)
print("Log Model R²:", r2_log)



Log Model MAE: 25051.55333867842
Log Model RMSE: 141148.35028490395
Log Model R²: 0.025280093101204937


In [20]:
comparison_log = pd.DataFrame({
    "actual_price": y_test_price.iloc[:5].values,
    "predicted_price": y_pred_price[:5]
})

comparison_log

,actual_price,predicted_price
0,28000.0,34245.678286
1,5900.0,17307.564585
2,41000.0,45487.190645
3,40250.0,51101.241502
4,77999.0,42867.669600


### Log-Target Linear Regression
Applying a log trnasformation to the target reduced MAE from approximately $30,132 to $25,052, but RMSE and R squared are not improved. This means the model still struggles with extreme high-price vehicles.

In [21]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators = 200,
                random_state = 42,
                n_jobs = -1
            )
        )
    ]
)

## create split data again
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state = 42
)

In [22]:
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [23]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = root_mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest MAE:", mae_rf)
print("Random Forest RMSE:", rmse_rf)
print("Random Forest R^2:", r2_rf)

Random Forest MAE: 21660.886068162923
Random Forest RMSE: 135095.54947887358
Random Forest R^2: 0.10708461313209672


Because the R^2 still small we cannot choose this model as a final model, we will add one more feature "model" into the input

In [24]:
features_v2 = [
    "model_year",
    "mileage",
    "brand",
    "model",
    "fuel_type",
    "accident",
    "clean_title",
    "transmission_group"
]

X_v2 = df[features_v2]

In [25]:
numeric_f_v2 = [
    "model_year",
    "mileage"
]

categorical_f_v2 = [
    "brand",
    "model",
    "fuel_type",
    "accident",
    "clean_title",
    "transmission_group"
]

In [28]:
# Create new preprocessor:
preprocessor_v2 = ColumnTransformer(
    transformers = [
        (
            "cat",
            OneHotEncoder(handle_unknown = "ignore"),
            categorical_f_v2
        ),
        (
            "num",
            "passthrough",
            numeric_f_v2
        )
    ]
)

In [26]:
#Split data for testing again
X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2,
    y,
    test_size=0.2,
    random_state=42
)

In [29]:
rf_model_v2 = Pipeline(
    steps = [
        ("preprocessor", preprocessor_v2),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators = 200,
                random_state = 42,
                n_jobs = -1
            )
        )
    ]
)

rf_model_v2.fit(X_train_v2,y_train_v2)

y_pred_rf_v2 = rf_model_v2.predict(X_test_v2)

In [30]:
mae_rf_v2 = mean_absolute_error(y_test_v2, y_pred_rf_v2)
rmse_rf_v2 = root_mean_squared_error(y_test_v2, y_pred_rf_v2)
r2_rf_v2 = r2_score(y_test_v2, y_pred_rf_v2)

print("Random Forest V2 MAE:", mae_rf_v2)
print("Random Forest V2 RMSE:", rmse_rf_v2)
print("Random Forest V2 R²:", r2_rf_v2)

Random Forest V2 MAE: 19675.907992518703
Random Forest V2 RMSE: 136697.47995886026
Random Forest V2 R²: 0.08578311237655689


In [31]:
y_train_pred_v2 = rf_model_v2.predict(X_train_v2)

train_mae = mean_absolute_error(y_train_v2, y_train_pred_v2)
train_rmse = root_mean_squared_error(y_train_v2, y_train_pred_v2)
train_r2 = r2_score(y_train_v2, y_train_pred_v2)

print("Train MAE:", train_mae)
print("Train RMSE:", train_rmse)
print("Train R²:", train_r2)

Train MAE: 4455.998142604719
Train RMSE: 9733.177670675905
Train R²: 0.96385666588469


These result shows this model is overfit.